# 06 — Generation: Your First RAG Chain (Runnables + Two-Node LangGraph)

**By the end of this notebook, you can build a grounded RAG chain with inline prompts + source citations, then rebuild it cleanly as a two-node LangGraph workflow.**

### Learning Objectives
- Author an inline strict grounded prompt (no external hub pulls) with `'say I don't know'` guardrails.
- Assemble a runnable LCEL chain (`|`) returning both answers and cited sources.
- Rebuild the exact pipeline as a typed two-node LangGraph (`retrieve -> generate`) and compare with vs without retrieval.

### Prerequisites
- Notebooks 00–05

## 1. Setup
Load `.env` configuration via `course/utils/models.py` and initialize our shared LLM and embeddings helpers.

In [1]:
# If running in Google Colab, uncomment: !git clone https://github.com/rahat4528/rag-from-scratch-course-v2.git && %cd rag-from-scratch-course-v2 && pip install -q -r course/requirements.txt
import sys
from pathlib import Path
repo_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'course' / 'utils').exists()), Path.cwd())
if str(repo_root) not in sys.path: sys.path.insert(0, str(repo_root))
from course.utils.models import get_llm, get_embeddings, resolve_course_root
course_dir = resolve_course_root(); data_dir = course_dir / 'data'
llm = get_llm(fallback_to_demo=True); embeddings = get_embeddings()
print(f'Ready | LLM={type(llm).__name__} | Embeddings={type(embeddings.base).__name__}')

[OfflineDemoChatModel] Notice: Provider 'ollama' not active (Ollama server is not reachable at http://localhost:11434. Install/start Ollama (`ollama pull llama3.2`) or switch RAG_PROVIDER in .env.). Running code-path demo mode.


W1112 00:48:51.634000 15736 .venv\Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Ready | LLM=OfflineDemoChatModel | Embeddings=LocalSentenceTransformerEmbeddings


## 2. Concept: From Linear Pipe (LCEL) to State Graph (LangGraph)

Notice our current LangChain docs pattern: instead of pulling brittle hub prompts, write prompts explicitly inline!

First we wire a classic **LCEL chain** (`|`), then rebuild it using **LangGraph `StateGraph`** — where each box (`retrieve`, `generate`) reads and updates a shared `TypedDict` state. Why learn both? Linear chains are quick; **LangGraph** lets us add grading, loops, and branches later!

```
LCEL Pipe  : Question --> {context: retriever | format, question: passthrough} --> Prompt --> LLM
LangGraph  : [START] --> [node: retrieve (updates state['docs'])] --> [node: generate (updates state['answer'])] --> [END]
```

## 3. Step-by-Step Build

### Step 1: Prepare Vector Retriever & Inline Grounded Prompt
Let's index our corpus, create a source-tagging formatter, and define our strict inline `ChatPromptTemplate`.

In [2]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from course.utils.helpers import load_corpus_documents

chunks = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=80).split_documents(load_corpus_documents(data_dir))
retriever = InMemoryVectorStore.from_documents(chunks, embeddings).as_retriever(search_kwargs={"k": 4})
format_docs = lambda docs: "\n\n".join(f"[{d.metadata['source']}]\n{d.page_content}" for d in docs)
prompt = ChatPromptTemplate.from_template(
    "Answer ONLY using the context below. If missing, say 'I don't know based on the provided context.'\n\n<context>\n{context}\n</context>\n\nQuestion: {question}"
)
        

> **What just happened?**
> Inline prompts stay version-controlled inside your repo rather than depending on external hub lookups.

### Step 2: Build the LCEL Runnable RAG Chain Returning Answer + Sources
Using `RunnableParallel` (`RunnablePassthrough.assign`), we run retrieval once and preserve source filenames right alongside the answer!

In [3]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

rag_with_sources = RunnablePassthrough.assign(docs=lambda x: retriever.invoke(x["question"])).assign(
    answer=lambda x: (prompt | llm | StrOutputParser()).invoke({"context": format_docs(x["docs"]), "question": x["question"]})
)
out = rag_with_sources.invoke({"question": "What is ZeroGravity Failover timeout threshold?"})
print("LCEL Answer :", out["answer"])
print("LCEL Sources:", sorted({d.metadata["source"] for d in out["docs"]}))
        

LCEL Answer : Based on the provided context: ### How does ZeroGravity Failover work, and which plan includes it? ZeroGravity Automatic Failover is exclusively included in the **Enterprise Plan** ($4,200/month). If a primary edge gateway or cluster coordinator becomes unresponsive for more than **40 milliseconds**, ZeroGravity automatically transfers fleet state ownership to a standby peer node without dropping active safety heartbeats. Enterprise Plan: $4,200 per month. Supports up to 200 active nodes. Includes ZeroGravity automatic failover, QuantumCache v4, 1-year telemetry retention, and 24/7/365 mission-critical support.
LCEL Sources: ['company_handbook.txt', 'product_faq.md']


> **What just happened?**
> Retrieving once inside `assign(docs=...)` avoids querying the vector store twice for citations vs prompt formatting.

### Step 3: Rebuild Cleanly as a Two-Node LangGraph (`retrieve -> generate`)
Now let's build the exact modern two-node **LangGraph** workflow (`START -> retrieve -> generate -> END`) over typed state!

In [4]:
from typing import List, TypedDict
from langchain_core.documents import Document
from langgraph.graph import START, END, StateGraph
class RAGState(TypedDict):
    question: str; docs: List[Document]; answer: str
def retrieve_step(state: RAGState): return {"docs": retriever.invoke(state["question"])}
def generate_step(state: RAGState):
    ctx = format_docs(state["docs"])
    return {"answer": (prompt | llm | StrOutputParser()).invoke({"context": ctx, "question": state["question"]})}
graph = StateGraph(RAGState).add_node("retrieve", retrieve_step).add_node("generate", generate_step)
rag_graph = graph.add_edge(START, "retrieve").add_edge("retrieve", "generate").add_edge("generate", END).compile()
        

> **What just happened?**
> Look how clean `StateGraph` is — every node simply returns a partial dict updating `RAGState`!

### Step 4: Compare Answers WITHOUT vs. WITH Retrieval (Plus Refusal Test)
Let's run the same questions through standalone `llm.invoke` vs our compiled `rag_graph` (including an unanswerable gap!).

In [5]:
for q in ["When does legacy REST API v1 stop working and what replaces it?", "What is Novasphere's NASDAQ stock ticker?"]:
    no_rag = llm.invoke(q).content
    res = rag_graph.invoke({"question": q, "docs": [], "answer": ""})
    srcs = sorted({d.metadata["source"] for d in res["docs"]})
    print(f"Q: {q}\n  No RAG  : {no_rag[:90]}\n  With RAG: {res['answer'][:120]}\n  Sources : {srcs}")
        

Q: When does legacy REST API v1 stop working and what replaces it?
  No RAG  : [OfflineDemoChatModel] Ready.
  With RAG: Based on the provided context: ### Important API Deprecation Notice **Legacy REST API v1** is scheduled for complete shu
  Sources : ['company_handbook.txt', 'product_faq.md', 'recent_changelog.md']
Q: What is Novasphere's NASDAQ stock ticker?
  No RAG  : [OfflineDemoChatModel] I cannot verify proprietary 2026 Novasphere Dynamics facts without 
  With RAG: I don't know based on the provided context. The supplied documents do not contain enough information to answer this ques
  Sources : ['company_handbook.txt', 'recent_changelog.md']


> **What just happened?**
> With RAG enabled, real company questions are grounded cleanly while unknown NASDAQ facts trigger explicit refusal!

## 4. Try It Yourself
1. **Test Zurich pet days** — *Hint:* Run `rag_graph.invoke({'question': 'When are pets allowed in Zurich?', 'docs': [], 'answer': ''})`.
2. **Inspect longest context token est** — *Hint:* Check `len(format_docs(res['docs'])) // 4` on `res`.

<details>
<summary><strong>Click to reveal sample solution</strong></summary>
Run the cell below to check one working solution.
</details>

In [6]:
r_pet = rag_graph.invoke({"question": "On which days are pets allowed in the Zurich office hub?", "docs": [], "answer": ""})
print("Pets Answer :", r_pet["answer"])
print("Context toks:", len(format_docs(r_pet["docs"])) // 4, "| Sources:", {d.metadata["source"] for d in r_pet["docs"]})
        

Pets Answer : Based on the provided context: Zurich Hub Pet Policy: Well-behaved dogs are permitted in the Zurich hub on Tuesdays and Thursdays only, provided they are registered with office facilities and have valid vaccinations on file. Pets are not permitted inside robotics testing bays.
Context toks: 335 | Sources: {'policy_brief.pdf', 'company_handbook.txt'}


## 5. Common Mistakes
- Pulling unpinned prompts from external hubs that change or break offline.
- Omitting `'say I don't know'` instructions in system/user prompt text.
- Running retrieval twice when returning both answer and citations.

## 6. Recap
- Inline `<context>` prompts keep behavior auditable and deterministic.
- LCEL chains wire quick straight-line apps; LangGraph provides explicit typed state for multi-step workflows.
- Comparing No-RAG vs RAG + testing unanswerables verifies real grounding before deployment.

## 7. Next Notebook
Next (`07_rag_with_your_own_files.ipynb`): Mini Capstone! Point the pipeline at your own folder of files, persist & reload indexes, and score our 14 eval questions!

## 8. Video Chapter Outline
1. **Grounded Prompting Best Practices** — XML delimiters + inline prompts.
2. **LCEL Runnable Assignment** — Retrieving once for answer + sources.
3. **Modern Two-Node LangGraph** — Building `retrieve -> generate` statefully.
4. **Before/After Comparison** — Standalone LLM vs grounded RAG vs gap refusal.
5. **Wrap-up** — Preparing for your own files in NB07.